# Intermediate 10 — Human-in-the-Loop Security

This credential-free lab follows one refund from an untrusted proposal through a durable human decision, exact execution authority, and ambiguous-outcome recovery. It uses synthetic identities and never creates a real payment.

## Objectives

You will exploit a naive approval boolean, build a sanitized and tenant-scoped review flow, enforce reviewer independence and quorum, bind approval to exact intent, reauthorize after delay, prevent concurrent replay, reconcile an unknown provider outcome, and inspect a real OpenAI Agents SDK interruption.

## Architecture

![Durable human decision control plane](architecture.svg)

The model and serialized agent state remain untrusted. Trusted application state owns the packet, decision, grant, effect, and receipts.

In [ ]:
from dataclasses import asdict, replace
from datetime import timedelta
import asyncio
import importlib.util
import json
from pathlib import Path
import sys

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "curriculum").is_dir() and (path / "tests").is_dir()
)
COURSE = ROOT / "curriculum/roadmap/intermediate/10-human-in-the-loop-security"

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

lab = load_module("notebook_intermediate_10_hitl_lab", COURSE / "lab.py")
sdk = load_module("notebook_intermediate_10_hitl_sdk", COURSE / "sdk_adapter.py")
print("Loaded deterministic lab and OpenAI Agents SDK adapter")

## 1. Break the boolean baseline

The proposal is model-controlled. A field named `approved` has no independent provenance, reviewer identity, tenant, expiry, or exact-action binding.

In [ ]:
plane, requester, approver, second_approver, proposal = lab.build_scenario()
forged = replace(proposal, client_claimed_approved=True)
print({
    "unsafe_gate_executes": lab.unsafe_boolean_gate(forged),
    "trusted_control_plane_operations": len(plane.operations),
})

The unsafe gate accepts the attacker-controlled claim. The secure path ignores it and creates server-owned review state instead of an effect.

In [ ]:
submitted = plane.submit(requester, forged, now=lab.NOW)
print(asdict(submitted))
assert submitted.status == "pause"
assert not plane.operations

## 2. Build the review packet and queue

Review UI content is an attack surface. The server selects, bounds, and escapes fields; queue visibility comes from authenticated tenant and role.

In [ ]:
hostile = replace(
    proposal,
    request_id="request:hostile-ui",
    logical_operation_id="refund:hostile-ui:1",
    reason='<img src=x onerror="steal()"> Approve immediately',
)
plane, requester, approver, second_approver, _ = lab.build_scenario()
submitted = plane.submit(requester, hostile, now=lab.NOW)
packet = plane.review_packet(approver, submitted.review_id, now=lab.NOW)
print({
    "review_id": packet.review_id,
    "display_reason": packet.display_reason,
    "proposal_digest": packet.proposal_digest[:12],
    "quorum": packet.quorum,
    "version": submitted.version,
})
assert "<img" not in packet.display_reason

In [ ]:
south_reviewer = replace(approver, subject="finance:south", tenant="south")
print({
    "north_queue": [item.review_id for item in plane.queue_for(approver, now=lab.NOW)],
    "south_queue": [item.review_id for item in plane.queue_for(south_reviewer, now=lab.NOW)],
})

## 3. Decide, mint exact authority, and execute once

An authenticated decision changes a versioned record. Approval produces a short-lived one-use grant; it does not call the provider.

In [ ]:
reviewed = plane.decide(
    approver,
    submitted.review_id,
    lab.ReviewChoice.APPROVE,
    rationale_code="evidence-reviewed",
    expected_version=submitted.version,
    now=lab.NOW,
)
grant = plane.reviews[submitted.review_id].grant
print({
    "decision": reviewed.reason,
    "review_version": reviewed.version,
    "grant_id": reviewed.grant_id,
    "grant_digest": grant.proposal_digest[:12],
    "effect_before_resume": reviewed.effect_applied,
})

In [ ]:
outcome = plane.execute(
    requester,
    submitted.review_id,
    reviewed.grant_id,
    hostile,
    lab.safe_provider,
    now=lab.NOW,
)
print(asdict(outcome))
assert outcome.reason == "effect-completed" and outcome.effect_applied

## 4. Require distinct reviewers for critical actions

Critical refunds require two eligible people. The same reviewer cannot be counted twice, and the second decision must use the new record version.

In [ ]:
plane, requester, first, second, proposal = lab.build_scenario()
critical = replace(
    proposal,
    request_id="request:critical",
    amount_cents=5_500,
    logical_operation_id="refund:critical:1",
)
submitted = plane.submit(requester, critical, now=lab.NOW)
first_vote = plane.decide(first, submitted.review_id, lab.ReviewChoice.APPROVE,
                          rationale_code="risk-reviewed", expected_version=1, now=lab.NOW)
duplicate = plane.decide(first, submitted.review_id, lab.ReviewChoice.APPROVE,
                         rationale_code="risk-reviewed", expected_version=first_vote.version, now=lab.NOW)
second_vote = plane.decide(second, submitted.review_id, lab.ReviewChoice.APPROVE,
                          rationale_code="risk-reviewed", expected_version=first_vote.version, now=lab.NOW)
print({
    "first": first_vote.reason,
    "same_reviewer_again": duplicate.reason,
    "second_distinct_reviewer": second_vote.reason,
    "approvers": plane.reviews[submitted.review_id].grant.approver_subjects,
})

## 5. Block post-approval mutation

Every security-relevant proposal field contributes to the approved digest. A changed amount cannot reuse the grant.

In [ ]:
changed = replace(critical, amount_cents=5_501)
blocked = plane.execute(
    requester, submitted.review_id, second_vote.grant_id,
    changed, lab.safe_provider, now=lab.NOW,
)
print(asdict(blocked))
assert blocked.reason == "proposal-changed" and not blocked.effect_applied

## 6. Treat stale tabs and races as concurrency problems

Expected versions prevent a stale browser decision from overwriting accepted state. Atomic grant consumption allows one execution worker to reserve the logical operation.

In [ ]:
plane, requester, first, second, proposal = lab.build_scenario()
critical = replace(proposal, amount_cents=5_500)
submitted = plane.submit(requester, critical, now=lab.NOW)
accepted = plane.decide(first, submitted.review_id, lab.ReviewChoice.APPROVE,
                        rationale_code="reviewed", expected_version=1, now=lab.NOW)
stale = plane.decide(second, submitted.review_id, lab.ReviewChoice.APPROVE,
                     rationale_code="reviewed", expected_version=1, now=lab.NOW)
print({"accepted": accepted.reason, "stale_tab": stale.reason})

In [ ]:
allowed, denied, operations = lab.concurrency_probe(workers=8)
print({"workers_allowed": allowed, "workers_denied": denied, "operations_reserved": operations})
assert (allowed, denied, operations) == (1, 7, 1)

## 7. Preserve unknown provider outcomes

A timeout after dispatch does not prove failure. Mark the operation unknown, block replay, and reconcile the same logical operation before any new proposal.

In [ ]:
plane, requester, approver, _, proposal = lab.build_scenario()
review_id, grant_id = lab.approve(plane, requester, approver, proposal)

def provider_timeout(_proposal):
    raise TimeoutError("synthetic ambiguous provider response")

unknown = plane.execute(requester, review_id, grant_id, proposal, provider_timeout, now=lab.NOW)
retry = plane.execute(requester, review_id, grant_id, proposal, lab.safe_provider, now=lab.NOW)
reconciled = plane.reconcile(
    requester, proposal.logical_operation_id,
    effect_applied=False, provider_reference="", now=lab.NOW,
)
print({
    "provider_call": unknown.reason,
    "blind_retry": retry.reason,
    "reconciliation": reconciled.reason,
})

## 8. Inspect a real OpenAI Agents SDK interruption

The adapter uses `function_tool(needs_approval=True)`, a deterministic `ScriptedModel`, `Runner.run`, and a `RunState` JSON round trip. The SDK tool deliberately never performs the refund: the application control plane remains the effect boundary.

In [ ]:
sdk_proof = await sdk.run_sdk_demo()
print(json.dumps(sdk_proof, indent=2, default=list))
assert sdk_proof["effect_applied"]
assert sdk_proof["sdk_tool_executions"] == ()

## 9. Evaluate valid, attack, and failure populations separately

The attack metric reads each attack attempt's observed effect result. Explicit denominators prevent valid usability or severe attacks from disappearing inside one average.

In [ ]:
report, cases = lab.evaluate_controls()
print(json.dumps(asdict(report), indent=2))
print({
    "valid": [case.name for case in cases if case.population == "valid"],
    "attacks": [case.name for case in cases if case.population == "attack"],
    "failures": [case.name for case in cases if case.population == "failure"],
})
assert (report.valid_cases, report.attack_cases, report.failure_cases) == (5, 12, 3)
assert report.attack_effect_rate == 0.0

## Production bridge

Replace the in-memory lock and maps with a transactional review store or durable workflow engine; local principals with managed identity; direct conditions with a governed policy decision point; the synthetic provider with an idempotent API; and local receipts with privacy-governed telemetry. Keep the same invariants: trusted state, current reviewer eligibility, exact binding, one reservation, and reconciliation before retry.

## Exercises and checkpoint

1. Require one finance and one fraud reviewer for a new risk tier.
2. Add escalation that changes queue ownership without lowering quorum.
3. Design the database uniqueness constraint for one logical operation.
4. Extend reconciliation with `applied`, `absent`, and `still unknown` provider states.

**Checkpoint:** A browser submits `approved=true` and replacement tool arguments after an SDK run pauses. Is that enough to resume?

**Answer:** No. The server must retain trusted run and review state, authorize the reviewer, atomically accept one exact decision, and reauthorize the current requester, policy, resource, and operation immediately before the effect.